In [0]:
# XGBoost — the machine learning model we'll use for churn prediction
# SHAP — explains why the model made each prediction (tells us which factors matter most for churn)
%pip install xgboost shap

In [0]:
# Notebook 4: Churn Prediction
# Food Delivery Analysis

# Importing all the libraries
# --------------------------------------------

from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load user features from Parquet
user_features = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/user_features.parquet"
)

print(f"Loaded {user_features.count():,} users with {len(user_features.columns)} features.")
user_features.display()

In [0]:
# Convert the PySpark dataframe to Pandas
# XGBoost and scikit-learn work with Pandas, not PySpark
user_pd = user_features.toPandas()

# Check the shape
print(f"Dataset shape: {user_pd.shape}")
print(f"\nChurn rate: {user_pd['churn_risk'].mean():.1%}")
print(f"\nColumn types:\n{user_pd.dtypes}")

In [0]:
# Drop user_id as it is just an identifier, not a feature
user_pd = user_pd.drop(columns=["user_id"])

# Convert text columns to numbers using label encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col_name in ["city", "payment_method", "top_cuisine"]:
    user_pd[col_name] = le.fit_transform(user_pd[col_name])

# Separate features and target variable
X = user_pd.drop(columns=["churn_risk"])
y = user_pd["churn_risk"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns:\n{list(X.columns)}")

In [0]:
# Split data into training and test sets
# 80% of the data is used to train the model
# 20% is kept aside to test how well the model performs on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]:,} users")
print(f"Test set size: {X_test.shape[0]:,} users")
print(f"\nChurn rate in training set: {y_train.mean():.1%}")
print(f"Churn rate in test set: {y_test.mean():.1%}")

In [0]:
# Train the XGBoost churn prediction model
# MLflow tracks everything automatically so we can review the results later

with mlflow.start_run(run_name="XGBoost Churn Prediction"):
    
    # Define the model and its settings
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    )
    
    # Train the model on the training set
    xgb_model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
    
    # Calculate performance scores
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log results to MLflow
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.xgboost.log_model(xgb_model, "xgboost_churn_model")
    
    print(f"Accuracy: {accuracy:.1%}")
    print(f"ROC AUC Score: {roc_auc:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn Risk"]))

In [0]:
# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn Risk"],
            yticklabels=["No Churn", "Churn Risk"])
plt.title("XGBoost Churn Prediction - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/tmp/xgb_confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved.")

In [0]:
# SHAP feature importance
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("XGBoost Churn Prediction - Feature Importance (SHAP)")
plt.tight_layout()
plt.savefig("/tmp/xgb_shap_importance.png", dpi=150)
plt.show()
print("SHAP feature importance saved.")

In [0]:
# LSTM Data Preparation
# Reshape user order history into monthly sequences

# Installing PyTorch instead of TensorFlow for LSTM implementation.
# TensorFlow has a known compatibility conflict with Python 3.12 and thevpre-installed Protobuf version in Databricks Serverless. 
# PyTorch resolves this cleanly and is the more widely adopted deep learning framework for custom LSTM work in production environments
# ----------------------------------------------------------------------------------------------------------------------------------------

%pip install torch xgboost shap 

In [0]:
# Imports for LSTM using PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.pytorch

print(f"PyTorch version: {torch.__version__}")

# Load the full dataset from Parquet
df_full = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/uae_food_delivery_750k.parquet"
)

# Convert to Pandas
df_pd = df_full.toPandas()

# Convert order_date to datetime and extract month
df_pd["order_date"] = pd.to_datetime(df_pd["order_date"])
df_pd["order_month"] = df_pd["order_date"].dt.month

print(f"Loaded {len(df_pd):,} rows")
print(f"Date range: {df_pd['order_date'].min()} to {df_pd['order_date'].max()}")

In [0]:
# Build monthly order sequences per user
# Each user gets 12 rows, one per month, with their activity for that month
# This is the format LSTM needs, a sequence of time steps per user

monthly = df_pd.groupby(["user_id", "order_month"]).agg(
    monthly_orders    = ("order_id", "count"),
    monthly_spend     = ("total_price_aed", "sum"),
    monthly_cancels   = ("order_status", lambda x: (x == "Cancelled").sum()),
    avg_risk_score    = ("order_quality_risk_score", "mean"),
    avg_delivery_time = ("delivery_duration_mins", "mean")
).reset_index()

# Pivot to get one row per user with 12 monthly columns per feature
# Then we reshape into a 3D array for LSTM input
all_users = df_pd["user_id"].unique()
churn_labels = df_pd.groupby("user_id")["churn_risk"].first().reset_index()

print(f"Monthly sequences built for {monthly['user_id'].nunique():,} users")
print(f"Months covered: {sorted(monthly['order_month'].unique())}")
print(f"\nSample monthly data:")
monthly.head(12).display()